**Modélisation:**
Persistance, régression linéaire et Random Forest. Séparation temporelle 80/20.

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("data/dataset_ml.csv", index_col="DATE_TIME", parse_dates=True)

cible = "AC_POWER_TOTAL"
features = [c for c in df.columns if c != cible]

# Séparation TEMPORELLE : les 80 % premiers jours pour entraîner, les 20 % derniers pour tester
split = int(len(df) * 0.8)
train = df.iloc[:split]
test = df.iloc[split:]

X_train, y_train = train[features], train[cible]
X_test, y_test = test[features], test[cible]

print("Train :", len(train), "| du", train.index.min().date(), "au", train.index.max().date())
print("Test  :", len(test), "| du", test.index.min().date(), "au", test.index.max().date())

Train : 1251 | du 2020-05-16 au 2020-06-11
Test  : 313 | du 2020-06-11 au 2020-06-17


In [9]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Prédiction naïve : la production reste celle du pas précédent
pred_persistance = X_test["prod_lag_1"]

rmse_p = np.sqrt(mean_squared_error(y_test, pred_persistance))
mae_p = mean_absolute_error(y_test, pred_persistance)

print(f"Persistance — RMSE : {rmse_p:.0f} kW | MAE : {mae_p:.0f} kW")

Persistance — RMSE : 3075 kW | MAE : 2228 kW


In [10]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)

rmse_lr = np.sqrt(mean_squared_error(y_test, pred_lr))
mae_lr = mean_absolute_error(y_test, pred_lr)

print(f"Régression linéaire — RMSE : {rmse_lr:.0f} kW | MAE : {mae_lr:.0f} kW")

Régression linéaire — RMSE : 614 kW | MAE : 409 kW


In [11]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)

rmse_rf = np.sqrt(mean_squared_error(y_test, pred_rf))
mae_rf = mean_absolute_error(y_test, pred_rf)

print(f"Random Forest — RMSE : {rmse_rf:.0f} kW | MAE : {mae_rf:.0f} kW")

Random Forest — RMSE : 656 kW | MAE : 428 kW


In [12]:
resultats = pd.DataFrame({
    "Modèle": ["Persistance", "Régression linéaire", "Random Forest"],
    "RMSE": [rmse_p, rmse_lr, rmse_rf],
    "MAE": [mae_p, mae_lr, mae_rf]
})
resultats["Skill score (%)"] = 100 * (1 - resultats["RMSE"] / rmse_p)
print(resultats.round(1))

                Modèle    RMSE     MAE  Skill score (%)
0          Persistance  3075.3  2228.4              0.0
1  Régression linéaire   613.5   408.7             80.0
2        Random Forest   656.4   428.1             78.7


In [13]:
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print(importances.head(10))

IRRADIATION           0.989811
azimut_soleil         0.000975
irr_moy_1h            0.000874
prod_moy_1h           0.000738
MODULE_TEMPERATURE    0.000736
irr_std_1h            0.000603
elevation_soleil      0.000570
irr_lag_4             0.000555
prod_veille           0.000553
ghi_ciel_clair        0.000540
dtype: float64


IRRADIATION concentre 99 % de l'importance : le modèle convertit l'irradiance mesurée au même instant en puissance. C'est du **nowcasting**, pas de la prévision.

Vérification : cible à +1 h, sans les variables mesurées au même instant.

In [16]:
# Cible = production exactement 1 h plus tard
df_prev = df.copy()
df_prev["cible_1h"] = df_prev[cible].reindex(df_prev.index + pd.Timedelta("1h")).values
df_prev = df_prev.dropna()

# Retirer les variables de l'instant présent qu'on n'aurait pas en vrai
features_prev = [c for c in features if c not in
                 ["IRRADIATION", "MODULE_TEMPERATURE", "AMBIENT_TEMPERATURE", "indice_ciel_clair"]]

split2 = int(len(df_prev) * 0.8)
Xtr, ytr = df_prev[features_prev].iloc[:split2], df_prev["cible_1h"].iloc[:split2]
Xte, yte = df_prev[features_prev].iloc[split2:], df_prev["cible_1h"].iloc[split2:]

rf2 = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf2.fit(Xtr, ytr)
pred2 = rf2.predict(Xte)

print(f"Prévision 1h — RMSE : {np.sqrt(mean_squared_error(yte, pred2)):.0f} kW")
rmse_pers_1h = np.sqrt(mean_squared_error(yte, df_prev[cible].iloc[split2:]))
print(f"Persistance 1h — RMSE : {rmse_pers_1h:.0f} kW")

Prévision 1h — RMSE : 3930 kW
Persistance 1h — RMSE : 5652 kW


Prévision à 1 h sans les variables simultanées : RMSE 3930 kW contre 5652 kW pour la persistance à 1 h → skill score 30,5 %. Le modèle prévoit grâce au profil horaire et à l'état récent de la centrale.

→ Notebook 06 : la météo (Open-Meteo) améliore-t-elle ce résultat ?